In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))


In [1]:
import pandas as pd
import numpy as np

In [3]:
from src.data import load_csv

In [4]:
df = load_csv('../data/hour.csv')

In [5]:
df

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,16
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,40
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,32
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,13
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,119
17375,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,89
17376,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,90
17377,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,61


In [6]:
class SimpleLinearRegression:
    def __init__(self):
        self.beta_0 = None
        self.beta_1 = None

    def fit(self, X:np.ndarray, y:np.ndarray):

        if len(X) != len(y):
            raise ValueError("X and y must have same number of samples")

        x_centered = X - np.mean(X)
        y_centered = y - np.mean(y)

        covariance = np.mean((x_centered * y_centered))
        variance = np.mean((x_centered)**2)

        if variance == 0:
            raise ZeroDivisionError("variance value found to be 0")

        self.beta_1 = covariance / variance

        self.beta_0 = np.mean(y) - (self.beta_1 * np.mean(X))

        return self

    def predict(self, X:np.array):

        if self.beta_0 is None or self.beta_1 is None:
            raise ValueError("Model is not fitted! Call .fit() first")


        return self.beta_0 + (self.beta_1 * X)

    def r2_score(self, X:np.ndarray, y:np.ndarray):

        y_pred = self.predict(X)

        ss_residual = np.sum((y - y_pred) ** 2)

        ss_total = np.sum((y - np.mean(y)) ** 2)

        if ss_total == 0:
            return 0.0

        return 1 - (ss_residual / ss_total)



### Finding simple linear regression between `atemp` and `cnt`

In [7]:
X = df['atemp']
y = df['cnt']

In [8]:
len(X)

17379

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [11]:
lr = SimpleLinearRegression()

In [12]:
print(len(X_train))
print(len(y_train))

13903
13903


In [13]:
lr.fit(X_train, y_train)

In [14]:
y_pred = lr.predict(x_test)
print(y_pred)

13903    287.366558
13904    281.237934
13905    287.366558
13906    281.237934
13907    275.149631
            ...    
17374     85.767096
17375     85.767096
17376     85.767096
17377     91.855400
17378     91.855400
Name: atemp, Length: 3476, dtype: float64


In [15]:
print(lr.r2_score(X_train, y_train)*100,"%")

18.192298413170192 %


In [16]:
print("R2 Score on test: ", lr.r2_score(x_test, y_test))

R2 Score on test:  0.009659326027157245


In [17]:
print(f"Value of beta_1 the slope: {lr.beta_1}")
print(f"Value of beta_0 the intercept: {lr.beta_0}")
print(f"Best fit line equation becomes: y={lr.beta_1:.2f}x + {lr.beta_0:.2f}")

Value of beta_1 the slope: 403.1989233221956
Value of beta_0 the intercept: -18.09694642403494
Best fit line equation becomes: y=403.20x + -18.10


In [27]:
## Polyfit:
beta_1, beta_0 = np.polyfit(X_train, y_train, 1)
print(f"Verified beta_1 the slope: , {beta_1}, {np.round(lr.beta_1) == np.round(beta_1)}")
print(f"Verified beta_0 theintercept: , {beta_0}, {np.round(lr.beta_0) == np.round(beta_0)}")

Verified beta_1 the slope: , 403.19892332219035, True
Verified beta_0 theintercept: , -18.096946424035277, True


As per my observation of the current SimpleRegression model: signle feature R2_Score is only 0.16 or 16% because our data needs preprocessing, the outliers and skewness along with high correlation are the factors that are deviating our best fit line from it's original slope and intercept. In the next few days we will observe why the model is failing and then at last use it on a cleaned dataset.

This notebook was for me to understand the mathematical intuition behind our favourite `Simple Linear Regression`.
Today I learnt that it is not just about fitting a line through the cloud of data points, but it is about calculating the accurate slope and intercept from our data, the un-processed data i.e data with outliers, skewness, correlation can hinder our prediction about the actual values, leading to a deviated best fit line and more errors in our prediction. With the help of numpy, I was able to derive the `.fit()` and `.predict()` function along with the `r2_score()` which is a metric for us to measure the accuracy of our predicted values using RSS and TSS. 

After turning `shuffle = False` the accuracy went up by 2% from 0.16 to 0.18.

### Today I learned:
1. Linear regression is not that complicated, it is easy to understand and interpret, but this makes it doable and not simple
2. How math behind SLR works, why slope and intercept matters while predicting y
3. How the error that is: true - predicted can help us minimise them using MAE, MSE, RMSE and how useful metrics like RSS and TSS are derived.
4. Leanred how OOP concepts like, objects, class and instance method work.
5. Why do we need to pre-process our data before feeding it to our model
6. I enjoyed defining functions and using them

Good Night BYEE!!